In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score

from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit import DataStructs


# ============================================================
# 1. 文件路径
# ============================================================

FULL_CSV = r"../outputs/recommendations_after_post_screening_full_model_ema_True_decor_True_topk_True.csv"
RANDOM_CSV = r"../outputs/recommendations_after_post_screening_random.csv"
MORGAN_CSV = r"../outputs/recommendations_after_post_screening_morgan_ema_True_decor_True_topk_True.csv"

WO_EMA_CSV = r"../outputs/recommendations_after_post_screening_full_model_ema_False_decor_True_topk_True.csv"
WO_DECOR_CSV = r"../outputs/recommendations_after_post_screening_full_model_ema_True_decor_False_topk_True.csv"
WO_TOPK_CSV = r"../outputs/recommendations_after_post_screening_full_model_ema_True_decor_True_topk_False.csv"

PCL_ENCODER_ONLY_CSV = r"../outputs/recommendations_after_post_screening_pcl_encoder_only_ema_True_decor_True_topk_True.csv"
USL_ENCODER_ONLY_CSV = r"../outputs/recommendations_after_post_screening_usl_encoder_only_ema_True_decor_True_topk_True.csv"

PCL_ENCODER_CLUSTERING_CSV = r"../outputs/recommendations_after_post_screening_pcl_encoder_clustering_ema_True_decor_True_topk_True.csv"
USL_ENCODER_CLUSTERING_CSV = r"../outputs/recommendations_after_post_screening_usl_encoder_clustering_ema_True_decor_True_topk_True.csv"


FULL_EMB = r"../outputs/embeddings_after_post_screening_full_model_ema_True_decor_True_topk_True.npy"
RANDOM_EMB = r"../outputs/embeddings_after_post_screening_random.npy"
MORGAN_EMB = r"../outputs/embeddings_after_post_screening_morgan_ema_True_decor_True_topk_True.npy"

WO_EMA_EMB = r"../outputs/embeddings_after_post_screening_full_model_ema_False_decor_True_topk_True.npy"
WO_DECOR_EMB = r"../outputs/embeddings_after_post_screening_full_model_ema_True_decor_False_topk_True.npy"
WO_TOPK_EMB = r"../outputs/embeddings_after_post_screening_full_model_ema_True_decor_True_topk_False.npy"

PCL_ENCODER_ONLY_EMB = r"../outputs/embeddings_after_post_screening_pcl_encoder_only_ema_True_decor_True_topk_True.npy"
USL_ENCODER_ONLY_EMB = r"../outputs/embeddings_after_post_screening_usl_encoder_only_ema_True_decor_True_topk_True.npy"

PCL_ENCODER_CLUSTERING_EMB = r"../outputs/embeddings_after_post_screening_pcl_encoder_clustering_ema_True_decor_True_topk_True.npy"
USL_ENCODER_CLUSTERING_EMB = r"../outputs/embeddings_after_post_screening_usl_encoder_clustering_ema_True_decor_True_topk_True.npy"


PROTO_EMB = r"../checkpoints_origin_backup/proto_centroids_GINE_epoch_300.pth"

# 你提供的已报道含硼添加剂 JSON 文件
KNOWN_ADDITIVE_JSON = r"../data/additives.json"

OUT_DIR = r"./"

SMILES_COL = "smiles"

FULL_BEFORE_N = 30800
RANDOM_BEFORE_N = 30800
MORGAN_BEFORE_N = 30800
WO_EMA_BEFORE_N = 30800
WO_DECOR_BEFORE_N = 30800
WO_TOPK_BEFORE_N = 30800
PCL_ENCODER_ONLY_BEFORE_N = 30800
USL_ENCODER_ONLY_BEFORE_N = 30800
PCL_ENCODER_CLUSTERING_BEFORE_N = 30800
USL_ENCODER_CLUSTERING_BEFORE_N = 30800


# ============================================================
# 2. 基础辅助函数
# ============================================================

def ensure_out_dir(out_dir):
    if not os.path.exists(out_dir):
        os.makedirs(out_dir)


def check_file_exists(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"找不到文件：{path}")


def load_data(csv_path, emb_path):
    check_file_exists(csv_path)
    check_file_exists(emb_path)

    df = pd.read_csv(csv_path)
    emb = np.load(emb_path)

    if len(df) != emb.shape[0]:
        raise ValueError(
            f"CSV 行数和 embedding 数量不一致：\n"
            f"{csv_path}: {len(df)} rows\n"
            f"{emb_path}: {emb.shape[0]} embeddings"
        )

    return df, emb


def load_prototype_centroids(pth_path):
    check_file_exists(pth_path)

    obj = torch.load(pth_path, map_location="cpu")

    if isinstance(obj, torch.Tensor):
        centroids = obj

    elif isinstance(obj, dict):
        possible_keys = [
            "prototype_centroids",
            "prototypes",
            "centroids",
            "proto_centroids",
            "prototype_embeddings",
            "cluster_centers",
            "proto_centroids_GINE"
        ]

        found_key = None
        for key in possible_keys:
            if key in obj:
                found_key = key
                break

        if found_key is None:
            raise KeyError(
                f"在 {pth_path} 中没有找到 prototype centroids。\n"
                f"当前 pth 文件包含的 keys 是：{list(obj.keys())}\n"
                f"请把真实 key 加入 possible_keys。"
            )

        centroids = obj[found_key]

    else:
        raise TypeError(f"不支持的 pth 文件格式：{type(obj)}")

    if isinstance(centroids, torch.Tensor):
        centroids = centroids.detach().cpu().numpy()
    elif isinstance(centroids, np.ndarray):
        pass
    else:
        centroids = np.array(centroids)

    if centroids.ndim != 2:
        raise ValueError(
            f"prototype_centroids 应该是二维数组 [num_prototypes, emb_dim]，"
            f"但现在 shape 是 {centroids.shape}"
        )

    return centroids


# ============================================================
# 3. Morgan fingerprint 工具函数
# ============================================================

def get_morgan_generator(radius=2, fp_size=2048):
    generator = rdFingerprintGenerator.GetMorganGenerator(
        radius=radius,
        fpSize=fp_size
    )
    return generator


def compute_morgan_fps(smiles_list, radius=2, fp_size=2048):
    generator = get_morgan_generator(radius=radius, fp_size=fp_size)

    fps = []
    valid_smiles = []
    invalid_smiles = []

    for smi in smiles_list:
        smi = str(smi)

        mol = Chem.MolFromSmiles(smi)

        if mol is None:
            fps.append(None)
            invalid_smiles.append(smi)
        else:
            fp = generator.GetFingerprint(mol)
            fps.append(fp)
            valid_smiles.append(smi)

    return fps, valid_smiles, invalid_smiles


# ============================================================
# 4. 读取已报道含硼添加剂 JSON
# ============================================================

def load_known_additives_from_json(json_path):
    """
    支持你提供的 JSON 格式：

    {
        "TMSB": {
            "smiles": "...",
            "hybridization": [...],
            "functional_groups": [...]
        },
        "TCEB": {
            "smiles": "...",
            ...
        }
    }

    返回：
    known_df:
        columns = [
            known_name,
            known_smiles,
            known_hybridization,
            known_functional_groups
        ]
    known_fps:
        有效已报道添加剂的 Morgan fingerprints
    """

    check_file_exists(json_path)

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    records = []

    for name, info in data.items():
        if not isinstance(info, dict):
            continue

        smiles = info.get("smiles", None)

        if smiles is None:
            continue

        hybridization = info.get("hybridization", [])
        functional_groups = info.get("functional_groups", [])

        if isinstance(hybridization, list):
            hybridization_text = "; ".join([str(x) for x in hybridization])
        else:
            hybridization_text = str(hybridization)

        if isinstance(functional_groups, list):
            functional_groups_text = "; ".join([str(x) for x in functional_groups])
        else:
            functional_groups_text = str(functional_groups)

        records.append(
            {
                "known_name": name,
                "known_smiles": smiles,
                "known_hybridization": hybridization_text,
                "known_functional_groups": functional_groups_text
            }
        )

    known_df = pd.DataFrame(records)

    if len(known_df) == 0:
        raise ValueError(
            f"在 {json_path} 中没有读取到任何包含 smiles 的已报道添加剂。"
        )

    fps, valid_smiles, invalid_smiles = compute_morgan_fps(
        known_df["known_smiles"].tolist()
    )

    valid_indices = [i for i, fp in enumerate(fps) if fp is not None]
    known_fps = [fps[i] for i in valid_indices]
    known_df_valid = known_df.iloc[valid_indices].reset_index(drop=True)

    if len(known_fps) == 0:
        raise ValueError("已报道添加剂中没有有效 SMILES，无法计算 known-additive similarity。")

    print(f"Known additives loaded from JSON: {len(known_df_valid)} valid")
    print(f"Invalid known additive SMILES: {len(invalid_smiles)}")

    return known_df_valid, known_fps


# ============================================================
# 5. Prototype 指标计算
# ============================================================

def compute_prototype_assignment(candidate_embeddings, prototype_centroids):
    Z = normalize(candidate_embeddings, axis=1)
    C = normalize(prototype_centroids, axis=1)

    sim_matrix = Z @ C.T

    assigned_proto = np.argmax(sim_matrix, axis=1)
    max_sim = np.max(sim_matrix, axis=1)

    sorted_sim = np.sort(sim_matrix, axis=1)

    if sim_matrix.shape[1] >= 2:
        second_sim = sorted_sim[:, -2]
    else:
        second_sim = np.full_like(max_sim, np.nan)

    similarity_gap = max_sim - second_sim

    return sim_matrix, max_sim, assigned_proto, second_sim, similarity_gap


def compute_prototype_coverage(assigned_proto, num_prototypes):
    covered = np.unique(assigned_proto)
    coverage_ratio = len(covered) / num_prototypes
    coverage_text = f"{len(covered)}/{num_prototypes}"

    return coverage_ratio, coverage_text, covered


def compute_assignment_entropy(assigned_proto, num_prototypes):
    counts = np.bincount(assigned_proto, minlength=num_prototypes)

    if counts.sum() == 0:
        probs = np.zeros_like(counts, dtype=float)
        return np.nan, np.nan, counts, probs

    probs = counts / counts.sum()
    probs_nonzero = probs[probs > 0]

    entropy = -np.sum(probs_nonzero * np.log(probs_nonzero))
    norm_entropy = entropy / np.log(num_prototypes) if num_prototypes > 1 else np.nan

    return entropy, norm_entropy, counts, probs


def compute_silhouette(candidate_embeddings, assigned_proto):
    unique_labels = np.unique(assigned_proto)

    if len(unique_labels) < 2 or len(unique_labels) >= len(assigned_proto):
        return np.nan

    Z = normalize(candidate_embeddings, axis=1)

    return silhouette_score(Z, assigned_proto, metric="cosine")


# ============================================================
# 6. 候选分子内部多样性
# ============================================================

def compute_candidate_diversity(smiles_list):
    fps, valid_smiles, invalid_smiles = compute_morgan_fps(smiles_list)

    fps = [fp for fp in fps if fp is not None]

    if len(fps) < 2:
        return np.nan, np.nan, len(valid_smiles), len(invalid_smiles)

    sims = []

    for i in range(len(fps)):
        for j in range(i + 1, len(fps)):
            sim = DataStructs.TanimotoSimilarity(fps[i], fps[j])
            sims.append(sim)

    mean_tanimoto_similarity = float(np.mean(sims))
    diversity = 1.0 - mean_tanimoto_similarity

    return diversity, mean_tanimoto_similarity, len(valid_smiles), len(invalid_smiles)


# ============================================================
# 7. 与已报道添加剂的相似度和 novelty score
# ============================================================

def compute_known_additive_similarity(smiles_list, known_df, known_fps):
    """
    对每个候选分子，计算它与所有已报道添加剂之间的最大 Tanimoto similarity。

    Known-additive similarity:
        max_j Tanimoto(candidate_i, known_additive_j)

    Novelty score:
        1 - Known-additive similarity

    同时输出最相近的已报道添加剂名称和 SMILES。
    """

    candidate_fps, valid_smiles, invalid_smiles = compute_morgan_fps(smiles_list)

    max_sims = []
    novelty_scores = []

    nearest_known_indices = []
    nearest_known_names = []
    nearest_known_smiles = []
    nearest_known_hybridization = []
    nearest_known_functional_groups = []

    for fp in candidate_fps:
        if fp is None:
            max_sims.append(np.nan)
            novelty_scores.append(np.nan)

            nearest_known_indices.append(np.nan)
            nearest_known_names.append(np.nan)
            nearest_known_smiles.append(np.nan)
            nearest_known_hybridization.append(np.nan)
            nearest_known_functional_groups.append(np.nan)
            continue

        sims = DataStructs.BulkTanimotoSimilarity(fp, known_fps)

        max_sim = float(np.max(sims))
        nearest_idx = int(np.argmax(sims))
        novelty = 1.0 - max_sim

        nearest_row = known_df.iloc[nearest_idx]

        max_sims.append(max_sim)
        novelty_scores.append(novelty)

        nearest_known_indices.append(nearest_idx)
        nearest_known_names.append(nearest_row["known_name"])
        nearest_known_smiles.append(nearest_row["known_smiles"])
        nearest_known_hybridization.append(nearest_row["known_hybridization"])
        nearest_known_functional_groups.append(nearest_row["known_functional_groups"])

    max_sims = np.array(max_sims, dtype=float)
    novelty_scores = np.array(novelty_scores, dtype=float)

    valid_max_sims = max_sims[~np.isnan(max_sims)]

    if len(valid_max_sims) == 0:
        summary = {
            "Known-additive similarity mean": np.nan,
            "Known-additive similarity std": np.nan,
            "Known-additive similarity median": np.nan,
            "Known-additive similarity min": np.nan,
            "Known-additive similarity max": np.nan,

            "Novelty score mean": np.nan,
            "Novelty score std": np.nan,
            "Novelty score median": np.nan,
            "Novelty score min": np.nan,
            "Novelty score max": np.nan,

            "High similarity ratio >0.75": np.nan,
            "Moderate similarity ratio 0.40-0.75": np.nan,
            "Low similarity ratio <0.40": np.nan,

            "Valid candidate SMILES for known similarity": 0,
            "Invalid candidate SMILES for known similarity": len(invalid_smiles),
        }

    else:
        valid_novelty = 1.0 - valid_max_sims

        summary = {
            "Known-additive similarity mean": float(np.mean(valid_max_sims)),
            "Known-additive similarity std": float(np.std(valid_max_sims)),
            "Known-additive similarity median": float(np.median(valid_max_sims)),
            "Known-additive similarity min": float(np.min(valid_max_sims)),
            "Known-additive similarity max": float(np.max(valid_max_sims)),

            "Novelty score mean": float(np.mean(valid_novelty)),
            "Novelty score std": float(np.std(valid_novelty)),
            "Novelty score median": float(np.median(valid_novelty)),
            "Novelty score min": float(np.min(valid_novelty)),
            "Novelty score max": float(np.max(valid_novelty)),

            "High similarity ratio >0.75": float(np.mean(valid_max_sims > 0.75)),
            "Moderate similarity ratio 0.40-0.75": float(
                np.mean((valid_max_sims >= 0.40) & (valid_max_sims <= 0.75))
            ),
            "Low similarity ratio <0.40": float(np.mean(valid_max_sims < 0.40)),

            "Valid candidate SMILES for known similarity": len(valid_smiles),
            "Invalid candidate SMILES for known similarity": len(invalid_smiles),
        }

    detail = {
        "known_additive_max_similarity": max_sims,
        "novelty_score": novelty_scores,
        "nearest_known_additive_index": nearest_known_indices,
        "nearest_known_additive_name": nearest_known_names,
        "nearest_known_additive_smiles": nearest_known_smiles,
        "nearest_known_additive_hybridization": nearest_known_hybridization,
        "nearest_known_additive_functional_groups": nearest_known_functional_groups,
    }

    return summary, detail


# ============================================================
# 8. 单个方法评估
# ============================================================

def evaluate_method(
    method_name,
    df,
    embeddings,
    prototype_centroids,
    smiles_col,
    before_screening_n=None,
    known_df=None,
    known_fps=None
):
    num_prototypes = prototype_centroids.shape[0]

    sim_matrix, max_sim, assigned_proto, second_sim, similarity_gap = compute_prototype_assignment(
        embeddings,
        prototype_centroids
    )

    coverage_ratio, coverage_text, covered = compute_prototype_coverage(
        assigned_proto,
        num_prototypes
    )

    entropy, norm_entropy, proto_counts, proto_probs = compute_assignment_entropy(
        assigned_proto,
        num_prototypes
    )

    silhouette = compute_silhouette(
        embeddings,
        assigned_proto
    )

    if smiles_col not in df.columns:
        raise ValueError(
            f"CSV 中找不到 SMILES 列：{smiles_col}\n"
            f"当前 CSV 列名为：{list(df.columns)}"
        )

    smiles_list = df[smiles_col].astype(str).tolist()

    diversity, mean_tanimoto, valid_smiles_num, invalid_smiles_num = compute_candidate_diversity(
        smiles_list
    )

    if known_df is not None and known_fps is not None:
        known_summary, known_detail = compute_known_additive_similarity(
            smiles_list=smiles_list,
            known_df=known_df,
            known_fps=known_fps
        )
    else:
        known_summary = {}
        known_detail = {
            "known_additive_max_similarity": np.full(len(df), np.nan),
            "novelty_score": np.full(len(df), np.nan),
            "nearest_known_additive_index": np.full(len(df), np.nan),
            "nearest_known_additive_name": np.full(len(df), np.nan),
            "nearest_known_additive_smiles": np.full(len(df), np.nan),
            "nearest_known_additive_hybridization": np.full(len(df), np.nan),
            "nearest_known_additive_functional_groups": np.full(len(df), np.nan),
        }

    survival_rate = len(df) / before_screening_n if before_screening_n else np.nan

    summary = {
        "Method": method_name,
        "Before screening number": before_screening_n if before_screening_n is not None else np.nan,
        "After screening number": len(df),
        "Post-screening survival rate": survival_rate,

        "Prototype similarity mean": float(np.mean(max_sim)),
        "Prototype similarity std": float(np.std(max_sim)),
        "Prototype similarity median": float(np.median(max_sim)),
        "Prototype similarity min": float(np.min(max_sim)),
        "Prototype similarity max": float(np.max(max_sim)),

        "Similarity gap mean": float(np.mean(similarity_gap)),
        "Similarity gap std": float(np.std(similarity_gap)),

        "Prototype coverage": coverage_text,
        "Prototype coverage ratio": float(coverage_ratio),

        "Assignment entropy": float(entropy),
        "Normalized assignment entropy": float(norm_entropy),

        "Candidate diversity": float(diversity),
        "Mean pairwise Tanimoto similarity": float(mean_tanimoto),

        "Known-additive similarity mean": known_summary.get("Known-additive similarity mean", np.nan),
        "Known-additive similarity std": known_summary.get("Known-additive similarity std", np.nan),
        "Known-additive similarity median": known_summary.get("Known-additive similarity median", np.nan),
        "Known-additive similarity min": known_summary.get("Known-additive similarity min", np.nan),
        "Known-additive similarity max": known_summary.get("Known-additive similarity max", np.nan),

        "Novelty score mean": known_summary.get("Novelty score mean", np.nan),
        "Novelty score std": known_summary.get("Novelty score std", np.nan),
        "Novelty score median": known_summary.get("Novelty score median", np.nan),
        "Novelty score min": known_summary.get("Novelty score min", np.nan),
        "Novelty score max": known_summary.get("Novelty score max", np.nan),

        "High similarity ratio >0.75": known_summary.get("High similarity ratio >0.75", np.nan),
        "Moderate similarity ratio 0.40-0.75": known_summary.get("Moderate similarity ratio 0.40-0.75", np.nan),
        "Low similarity ratio <0.40": known_summary.get("Low similarity ratio <0.40", np.nan),

        "Valid candidate SMILES for known similarity": known_summary.get(
            "Valid candidate SMILES for known similarity", np.nan
        ),
        "Invalid candidate SMILES for known similarity": known_summary.get(
            "Invalid candidate SMILES for known similarity", np.nan
        ),

        "Silhouette score": float(silhouette),

        "Valid SMILES number": valid_smiles_num,
        "Invalid SMILES number": invalid_smiles_num,
    }

    for i in range(num_prototypes):
        summary[f"Prototype {i + 1} count"] = int(proto_counts[i])
        summary[f"Prototype {i + 1} ratio"] = float(proto_probs[i])

    detail_df = df.copy()
    detail_df["method"] = method_name

    detail_df["assigned_prototype"] = assigned_proto + 1
    detail_df["max_prototype_similarity"] = max_sim
    detail_df["second_prototype_similarity"] = second_sim
    detail_df["similarity_gap"] = similarity_gap

    detail_df["known_additive_max_similarity"] = known_detail["known_additive_max_similarity"]
    detail_df["novelty_score"] = known_detail["novelty_score"]
    detail_df["nearest_known_additive_index"] = known_detail["nearest_known_additive_index"]
    detail_df["nearest_known_additive_name"] = known_detail["nearest_known_additive_name"]
    detail_df["nearest_known_additive_smiles"] = known_detail["nearest_known_additive_smiles"]
    detail_df["nearest_known_additive_hybridization"] = known_detail["nearest_known_additive_hybridization"]
    detail_df["nearest_known_additive_functional_groups"] = known_detail["nearest_known_additive_functional_groups"]

    for i in range(num_prototypes):
        detail_df[f"prototype_{i + 1}_similarity"] = sim_matrix[:, i]

    return summary, detail_df


# ============================================================
# 9. 生成主文表格
# ============================================================

def make_paper_table(summary_df):
    paper_df = summary_df.copy()

    paper_df["Prototype similarity ↑"] = paper_df.apply(
        lambda row: f"{row['Prototype similarity mean']:.4f} ± {row['Prototype similarity std']:.4f}",
        axis=1
    )

    paper_df["Prototype coverage ↑"] = paper_df["Prototype coverage"]

    paper_df["Assignment entropy ↑"] = paper_df["Normalized assignment entropy"].apply(
        lambda x: f"{x:.4f}" if pd.notna(x) else "nan"
    )

    paper_df["Known-additive similarity"] = paper_df.apply(
        lambda row: f"{row['Known-additive similarity mean']:.4f} ± {row['Known-additive similarity std']:.4f}"
        if pd.notna(row["Known-additive similarity mean"]) else "nan",
        axis=1
    )

    paper_df["Novelty score ↑"] = paper_df.apply(
        lambda row: f"{row['Novelty score mean']:.4f} ± {row['Novelty score std']:.4f}"
        if pd.notna(row["Novelty score mean"]) else "nan",
        axis=1
    )

    paper_df["Moderate novelty ratio"] = paper_df["Moderate similarity ratio 0.40-0.75"].apply(
        lambda x: f"{x * 100:.2f}%" if pd.notna(x) else "nan"
    )

    paper_df["High similarity ratio"] = paper_df["High similarity ratio >0.75"].apply(
        lambda x: f"{x * 100:.2f}%" if pd.notna(x) else "nan"
    )

    paper_df["Low similarity ratio"] = paper_df["Low similarity ratio <0.40"].apply(
        lambda x: f"{x * 100:.2f}%" if pd.notna(x) else "nan"
    )

    paper_df["Candidate diversity ↑"] = paper_df["Candidate diversity"].apply(
        lambda x: f"{x:.4f}" if pd.notna(x) else "nan"
    )

    paper_df["Post-screening survival ↑"] = paper_df["Post-screening survival rate"].apply(
        lambda x: f"{x * 100:.4f}%" if pd.notna(x) else "nan"
    )

    paper_df["Silhouette ↑"] = paper_df["Silhouette score"].apply(
        lambda x: f"{x:.4f}" if pd.notna(x) else "nan"
    )

    paper_df["After screening number"] = paper_df["After screening number"].astype(int)

    final_table = paper_df[
        [
            "Method",
            "After screening number",
            "Prototype similarity ↑",
            "Prototype coverage ↑",
            "Assignment entropy ↑",
            "Known-additive similarity",
            "Novelty score ↑",
            "Moderate novelty ratio",
            "Candidate diversity ↑",
            "Post-screening survival ↑",
            "Silhouette ↑"
        ]
    ]

    return final_table


def make_extended_paper_table(summary_df):
    """
    这个表比 main_text_ablation_table 更完整，
    适合放 Supplementary Information。
    """

    paper_df = summary_df.copy()

    paper_df["Prototype similarity ↑"] = paper_df.apply(
        lambda row: f"{row['Prototype similarity mean']:.4f} ± {row['Prototype similarity std']:.4f}",
        axis=1
    )

    paper_df["Known-additive similarity"] = paper_df.apply(
        lambda row: f"{row['Known-additive similarity mean']:.4f} ± {row['Known-additive similarity std']:.4f}"
        if pd.notna(row["Known-additive similarity mean"]) else "nan",
        axis=1
    )

    paper_df["Novelty score ↑"] = paper_df.apply(
        lambda row: f"{row['Novelty score mean']:.4f} ± {row['Novelty score std']:.4f}"
        if pd.notna(row["Novelty score mean"]) else "nan",
        axis=1
    )

    paper_df["Assignment entropy ↑"] = paper_df["Normalized assignment entropy"].apply(
        lambda x: f"{x:.4f}" if pd.notna(x) else "nan"
    )

    paper_df["Candidate diversity ↑"] = paper_df["Candidate diversity"].apply(
        lambda x: f"{x:.4f}" if pd.notna(x) else "nan"
    )

    paper_df["Post-screening survival ↑"] = paper_df["Post-screening survival rate"].apply(
        lambda x: f"{x * 100:.4f}%" if pd.notna(x) else "nan"
    )

    paper_df["Silhouette ↑"] = paper_df["Silhouette score"].apply(
        lambda x: f"{x:.4f}" if pd.notna(x) else "nan"
    )

    paper_df["High similarity ratio >0.75"] = paper_df["High similarity ratio >0.75"].apply(
        lambda x: f"{x * 100:.2f}%" if pd.notna(x) else "nan"
    )

    paper_df["Moderate similarity ratio 0.40-0.75"] = paper_df[
        "Moderate similarity ratio 0.40-0.75"
    ].apply(
        lambda x: f"{x * 100:.2f}%" if pd.notna(x) else "nan"
    )

    paper_df["Low similarity ratio <0.40"] = paper_df["Low similarity ratio <0.40"].apply(
        lambda x: f"{x * 100:.2f}%" if pd.notna(x) else "nan"
    )

    final_table = paper_df[
        [
            "Method",
            "After screening number",
            "Prototype similarity ↑",
            "Prototype coverage",
            "Assignment entropy ↑",
            "Known-additive similarity",
            "Novelty score ↑",
            "High similarity ratio >0.75",
            "Moderate similarity ratio 0.40-0.75",
            "Low similarity ratio <0.40",
            "Candidate diversity ↑",
            "Post-screening survival ↑",
            "Silhouette ↑"
        ]
    ]

    return final_table


def save_detail_csv(detail_df, method_name):
    safe_name = (
        method_name
        .lower()
        .replace(" ", "_")
        .replace("/", "without_")
        .replace("-", "_")
    )

    out_path = os.path.join(
        OUT_DIR,
        f"{safe_name}_postscreen_with_metrics.csv"
    )

    detail_df.to_csv(out_path, index=False)




# ============================================================
# 10. Radar plot for main-text ablation results
# ============================================================

def minmax_normalize(values, higher_is_better=True):
    """
    将不同量纲的指标归一化到 [0, 1]，用于 radar plot。

    higher_is_better=True:
        数值越大越好

    higher_is_better=False:
        数值越小越好，会自动反向归一化
    """

    values = np.asarray(values, dtype=float)

    if np.all(np.isnan(values)):
        return np.full_like(values, np.nan, dtype=float)

    v_min = np.nanmin(values)
    v_max = np.nanmax(values)

    if np.isclose(v_max, v_min):
        norm_values = np.ones_like(values, dtype=float)
    else:
        norm_values = (values - v_min) / (v_max - v_min)

    if not higher_is_better:
        norm_values = 1.0 - norm_values

    return norm_values


def rank_normalize(values, higher_is_better=True):
    """
    将指标转化为 rank score，用于 radar plot。
    排名第 1 的方法得 1，排名最后的方法得 0。

    higher_is_better=True:
        数值越大排名越靠前。
    """

    values = np.asarray(values, dtype=float)

    if np.all(np.isnan(values)):
        return np.full_like(values, np.nan, dtype=float)

    valid_mask = ~np.isnan(values)
    n = int(np.sum(valid_mask))

    if n <= 1:
        return np.ones_like(values, dtype=float)

    ranks = pd.Series(values).rank(
        ascending=not higher_is_better,
        method="min"
    ).values

    rank_scores = 1.0 - (ranks - 1.0) / (n - 1.0)
    rank_scores[~valid_mask] = np.nan

    return rank_scores


def plot_main_text_radar(summary_df, out_dir):
    """
    根据 main-text ablation table 中的核心指标绘制 radar plot。

    说明：
    1) 这里使用 summary_df 中的原始数值，而不是 paper_table 中的字符串格式结果。
    2) Novelty score 不进入 radar 图。
    3) Assignment entropy 不直接按“越大越好”处理，而是构造：
       Entropy-weighted assignment score =
       Normalized assignment entropy × Prototype similarity mean
       然后用 rank normalization 表示该综合指标的排名。
    """

    radar_df = summary_df.copy()

    # Moderate novelty ratio:
    # 候选分子与已知添加剂的最大 Tanimoto similarity 位于 0.40-0.75 的比例
    radar_df["Moderate novelty ratio"] = radar_df["Moderate similarity ratio 0.40-0.75"]

    # Assignment entropy 本身只表示 prototype 分布均匀性。
    # 为避免 random/Morgan 由于随机分散而获得高 entropy，
    # 这里将 entropy 与 prototype similarity 结合，表示“宽分布 + 高 prototype 置信度”。
    radar_df["Entropy-weighted assignment score"] = (
        radar_df["Normalized assignment entropy"]
        * radar_df["Prototype similarity mean"]
    )

    metrics = [
        {
            "column": "Prototype similarity mean",
            "label": "Prototype\nsimilarity",
            "normalize": "higher",
        },
        {
            "column": "Prototype coverage ratio",
            "label": "Prototype\ncoverage",
            "normalize": "higher",
        },
        {
            "column": "Entropy-weighted assignment score",
            "label": "Entropy-weighted\nassignment rank",
            "normalize": "rank",
        },
        {
            "column": "Moderate novelty ratio",
            "label": "Moderate\nnovelty ratio",
            "normalize": "higher",
        },
        {
            "column": "Candidate diversity",
            "label": "Candidate\ndiversity",
            "normalize": "higher",
        },
        {
            "column": "Silhouette score",
            "label": "Silhouette",
            "normalize": "higher",
        },
    ]

    missing_cols = [m["column"] for m in metrics if m["column"] not in radar_df.columns]
    if len(missing_cols) > 0:
        raise ValueError(f"summary_df 中缺少这些列，无法绘制 radar plot: {missing_cols}")

    plot_data = pd.DataFrame()
    plot_data["Method"] = radar_df["Method"]

    raw_metric_data = pd.DataFrame()
    raw_metric_data["Method"] = radar_df["Method"]

    for m in metrics:
        col = m["column"]
        values = radar_df[col].values

        raw_metric_data[col] = values

        if m["normalize"] == "higher":
            plot_data[m["label"]] = minmax_normalize(
                values,
                higher_is_better=True
            )

        elif m["normalize"] == "rank":
            plot_data[m["label"]] = rank_normalize(
                values,
                higher_is_better=True
            )

        else:
            raise ValueError(f"未知 normalize 类型: {m['normalize']}")

    labels = [m["label"] for m in metrics]
    num_vars = len(labels)

    angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(8.0, 8.0), subplot_kw=dict(polar=True))

    for _, row in plot_data.iterrows():
        method = row["Method"]
        values = row[labels].astype(float).values.tolist()
        values += values[:1]

        if method == "Full model":
            ax.plot(angles, values, linewidth=3.0, label=method)
            ax.fill(angles, values, alpha=0.10)
        else:
            ax.plot(angles, values, linewidth=1.6, alpha=0.75, label=method)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=10)

    ax.set_ylim(0, 1.0)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(["0.2", "0.4", "0.6", "0.8", "1.0"], fontsize=8)
    ax.grid(True, linestyle="--", linewidth=0.7, alpha=0.6)

    # 正文图一般不建议直接写 Main-text ablation radar plot 作为标题；
    # 标题信息建议放在 figure caption 中。
    # 如需标题，可以取消下一行注释。
    # ax.set_title("Ablation performance profile", fontsize=13, pad=24)

    ax.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.10),
        ncol=3,
        frameon=False,
        fontsize=9
    )

    plt.tight_layout()

    png_path = os.path.join(out_dir, "main_text_ablation_radar.png")
    pdf_path = os.path.join(out_dir, "main_text_ablation_radar.pdf")
    svg_path = os.path.join(out_dir, "main_text_ablation_radar.svg")

    plt.savefig(png_path, dpi=600, bbox_inches="tight", pad_inches=0.35)
    plt.savefig(pdf_path, bbox_inches="tight", pad_inches=0.35)
    plt.savefig(svg_path, bbox_inches="tight", pad_inches=0.35)
    plt.close()

    radar_csv_path = os.path.join(out_dir, "main_text_ablation_radar_normalized_data.csv")
    raw_csv_path = os.path.join(out_dir, "main_text_ablation_radar_raw_metric_data.csv")

    plot_data.to_csv(radar_csv_path, index=False)
    raw_metric_data.to_csv(raw_csv_path, index=False)

    # print("\nRadar plot saved:")
    # print(png_path)
    # print(pdf_path)
    # print(svg_path)
    # print(radar_csv_path)
    # print(raw_csv_path)


# ============================================================
# 10. 主程序
# ============================================================

def main():
    ensure_out_dir(OUT_DIR)

    print("Loading prototype centroids...")
    prototype_centroids = load_prototype_centroids(PROTO_EMB)
    print("Prototype centroids shape:", prototype_centroids.shape)

    print("\nLoading known reported boron additives...")
    known_df, known_fps = load_known_additives_from_json(KNOWN_ADDITIVE_JSON)
    print("Known additives dataframe shape:", known_df.shape)

    known_df.to_csv(
        os.path.join(OUT_DIR, "known_reported_additives_valid.csv"),
        index=False
    )

    methods = [
        {
            "name": "Full model",
            "csv": FULL_CSV,
            "emb": FULL_EMB,
            "before_n": FULL_BEFORE_N,
        },
        {
            "name": "Random selection",
            "csv": RANDOM_CSV,
            "emb": RANDOM_EMB,
            "before_n": RANDOM_BEFORE_N,
        },
        {
            "name": "Morgan fingerprints",
            "csv": MORGAN_CSV,
            "emb": MORGAN_EMB,
            "before_n": MORGAN_BEFORE_N,
        },
        {
            "name": "PCL encoder only",
            "csv": PCL_ENCODER_ONLY_CSV,
            "emb": PCL_ENCODER_ONLY_EMB,
            "before_n": PCL_ENCODER_ONLY_BEFORE_N,
        },
        {
            "name": "USL encoder only",
            "csv": USL_ENCODER_ONLY_CSV,
            "emb": USL_ENCODER_ONLY_EMB,
            "before_n": USL_ENCODER_ONLY_BEFORE_N,
        },
        {
            "name": "PCL encoder clustering",
            "csv": PCL_ENCODER_CLUSTERING_CSV,
            "emb": PCL_ENCODER_CLUSTERING_EMB,
            "before_n": PCL_ENCODER_CLUSTERING_BEFORE_N,
        },
        {
            "name": "USL encoder clustering",
            "csv": USL_ENCODER_CLUSTERING_CSV,
            "emb": USL_ENCODER_CLUSTERING_EMB,
            "before_n": USL_ENCODER_CLUSTERING_BEFORE_N,
        },
        {
            "name": "w/o EMA",
            "csv": WO_EMA_CSV,
            "emb": WO_EMA_EMB,
            "before_n": WO_EMA_BEFORE_N,
        },
        {
            "name": "w/o decorrelation",
            "csv": WO_DECOR_CSV,
            "emb": WO_DECOR_EMB,
            "before_n": WO_DECOR_BEFORE_N,
        },
        {
            "name": "w/o top-K",
            "csv": WO_TOPK_CSV,
            "emb": WO_TOPK_EMB,
            "before_n": WO_TOPK_BEFORE_N,
        },
    ]

    all_summaries = []
    all_details = []

    print("\nEvaluating methods...")

    for item in methods:
        method_name = item["name"]
        csv_path = item["csv"]
        emb_path = item["emb"]
        before_n = item["before_n"]

        # print(f"\nLoading {method_name}...")
        df, embeddings = load_data(csv_path, emb_path)

        if embeddings.shape[1] != prototype_centroids.shape[1]:
            raise ValueError(
                f"{method_name} embedding dim 和 prototype dim 不一致："
                f"{embeddings.shape[1]} vs {prototype_centroids.shape[1]}"
            )

        summary, detail_df = evaluate_method(
            method_name=method_name,
            df=df,
            embeddings=embeddings,
            prototype_centroids=prototype_centroids,
            smiles_col=SMILES_COL,
            before_screening_n=before_n,
            known_df=known_df,
            known_fps=known_fps
        )

        all_summaries.append(summary)
        all_details.append(detail_df)

        save_detail_csv(detail_df, method_name)

    summary_df = pd.DataFrame(all_summaries)

    paper_table = make_paper_table(summary_df)
    extended_table = make_extended_paper_table(summary_df)

    summary_df.to_csv(
        os.path.join(OUT_DIR, "postscreen_ablation_summary.csv"),
        index=False
    )

    summary_df.to_excel(
        os.path.join(OUT_DIR, "postscreen_ablation_summary.xlsx"),
        index=False
    )

    paper_table.to_csv(
        os.path.join(OUT_DIR, "main_text_ablation_table.csv"),
        index=False
    )

    paper_table.to_excel(
        os.path.join(OUT_DIR, "main_text_ablation_table.xlsx"),
        index=False
    )

    extended_table.to_csv(
        os.path.join(OUT_DIR, "extended_ablation_table_with_novelty.csv"),
        index=False
    )

    extended_table.to_excel(
        os.path.join(OUT_DIR, "extended_ablation_table_with_novelty.xlsx"),
        index=False
    )

    all_detail_df = pd.concat(all_details, axis=0, ignore_index=True)

    all_detail_df.to_csv(
        os.path.join(OUT_DIR, "all_methods_postscreen_with_metrics.csv"),
        index=False
    )

    all_detail_df.to_excel(
        os.path.join(OUT_DIR, "all_methods_postscreen_with_metrics.xlsx"),
        index=False
    )

    plot_main_text_radar(summary_df, OUT_DIR)

    print("\nDone! Main-text ablation table:")
    print(paper_table.to_string(index=False))

    print("\nDone! Extended ablation table:")
    print(extended_table.to_string(index=False))

    # print("\nSaved files:")
    # print(os.path.join(OUT_DIR, "known_reported_additives_valid.csv"))
    # print(os.path.join(OUT_DIR, "postscreen_ablation_summary.csv"))
    # print(os.path.join(OUT_DIR, "postscreen_ablation_summary.xlsx"))
    # print(os.path.join(OUT_DIR, "main_text_ablation_table.csv"))
    # print(os.path.join(OUT_DIR, "main_text_ablation_table.xlsx"))
    # print(os.path.join(OUT_DIR, "extended_ablation_table_with_novelty.csv"))
    # print(os.path.join(OUT_DIR, "extended_ablation_table_with_novelty.xlsx"))
    # print(os.path.join(OUT_DIR, "all_methods_postscreen_with_metrics.csv"))
    # print(os.path.join(OUT_DIR, "all_methods_postscreen_with_metrics.xlsx"))


if __name__ == "__main__":
    main()

Loading prototype centroids...
Prototype centroids shape: (7, 128)

Loading known reported boron additives...
Known additives loaded from JSON: 126 valid
Invalid known additive SMILES: 0
Known additives dataframe shape: (126, 4)

Evaluating methods...

Done! Main-text ablation table:
                Method  After screening number Prototype similarity ↑ Prototype coverage ↑ Assignment entropy ↑ Known-additive similarity Novelty score ↑ Moderate novelty ratio Candidate diversity ↑ Post-screening survival ↑ Silhouette ↑
            Full model                      71        0.7898 ± 0.0306                  5/7               0.7232           0.4025 ± 0.2446 0.5975 ± 0.2446                 25.35%                0.8982                   0.2305%       0.9761
      Random selection                     647        0.5217 ± 0.1478                  7/7               0.5297           0.3565 ± 0.0910 0.6435 ± 0.0910                 24.42%                0.7830                   2.1006%       0.5965
 